In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
from IPython.display import display

In [3]:
df = pd.read_csv('data/pre-thin-data.csv')
df['status'] = df['pre_HT'].apply(lambda x: 'Alive' if pd.notnull(x) and x > 0 else 'Dead')

### Combined Strategies with Composite Score

In [10]:
def k_row_thinning(df, k:int, start_row:int=1, row_col='Row', status_col='status'):

    assert k >= 2 and 1 <= start_row <= k
    d = df.copy()
    d['thin_decision'] = 'Dead (ignored)'
    alive = d[status_col].eq('Alive')
    rows_to_thin = ((d[row_col] - start_row) % k == 0)
    d.loc[alive & rows_to_thin, 'thin_decision'] = 'Thin'
    d.loc[alive & ~rows_to_thin, 'thin_decision'] = 'Keep'
    return d

three_row_thinning = lambda df, start_row=1: k_row_thinning(df, 3, start_row)
four_row_thinning  = lambda df, start_row=1: k_row_thinning(df, 4, start_row)
five_row_thinning  = lambda df, start_row=1: k_row_thinning(df, 5, start_row)

def stand_analysis(df_thinned, metric='pre_DBH', vol_col='pre_stem_vol', status_col='status'):
    alive   = df_thinned[df_thinned[status_col] == 'Alive'].copy()
    kept    = alive[alive['thin_decision'] == 'Keep']
    removed = alive[alive['thin_decision'] == 'Thin']

    # volumes
    pre_total_vol     = float(alive[vol_col].sum())   if len(alive) else 0.0
    post_total_vol    = float(kept[vol_col].sum())    if len(kept)  else 0.0
    removed_total_vol = float(removed[vol_col].sum()) if len(removed) else 0.0

    # central tendencies
    pre_median  = float(alive[metric].median()) if len(alive) else np.nan
    pre_mean    = float(alive[metric].mean())   if len(alive) else np.nan
    post_median = float(kept[metric].median())  if len(kept)  else np.nan
    post_mean   = float(kept[metric].mean())    if len(kept)  else np.nan
    d_median    = (post_median - pre_median) if (not np.isnan(post_median) and not np.isnan(pre_median)) else np.nan
    d_mean      = (post_mean   - pre_mean)   if (not np.isnan(post_mean)   and not np.isnan(pre_mean))   else np.nan

    # quartiles from pre-alive
    q1 = alive[metric].quantile(0.25) if len(alive) else np.nan
    q3 = alive[metric].quantile(0.75) if len(alive) else np.nan

    # counts by quartile (pre vs post)
    L_pre  = int((alive [metric] <= q1).sum()) if len(alive) else 0  # Q1
    H_pre  = int((alive [metric] >= q3).sum()) if len(alive) else 0  # Q4
    L_post = int((kept  [metric] <= q1).sum()) if len(kept)  else 0
    H_post = int((kept  [metric] >= q3).sum()) if len(kept)  else 0

    # removed by quartile
    L_cut = L_pre - L_post
    H_cut = H_pre - H_post

    # volume by Q4 removed
    removed_q4_vol = float(removed.loc[removed[metric] >= q3, vol_col].sum()) if len(removed) else 0.0
    pct_removed_vol_from_q4 = (100 * removed_q4_vol / removed_total_vol) if removed_total_vol > 0 else np.nan

    # key ratios
    r_q4 = (H_post / H_pre) if H_pre > 0 else np.nan              # Q4 retention ratio
    r_q1 = (L_cut / L_pre) if L_pre > 0 else np.nan               # Q1 removal ratio

    return {
        # overview
        '% trees removed': round((len(removed) / len(alive)) * 100, 2) if len(alive) else np.nan,
        '% volume removed': round((removed_total_vol / pre_total_vol) * 100, 2) if pre_total_vol > 0 else np.nan,
        'Trees kept': int(len(kept)),
        'Trees removed': int(len(removed)),

        # quartile event columns
        'Q1 cut (count)': int(L_cut),
        'Q4 cut (count)': int(H_cut),
        'Q4 remaining (count)': int(H_post),
        'Removed volume from Q4': round(removed_q4_vol, 2),
        '% of removed volume from Q4': round(pct_removed_vol_from_q4, 2) if not np.isnan(pct_removed_vol_from_q4) else np.nan,

        # ratios
        'Q1 removal ratio (removed/pre)': round(r_q1, 3) if not np.isnan(r_q1) else np.nan,
        'Q4 retention ratio (post/pre)': round(r_q4, 3) if not np.isnan(r_q4) else np.nan,

        'Q1 pre (count)': int(L_pre),
        'Q4 pre (count)': int(H_pre),

        # DBH/volume stats
        'Post-thinning total volume': round(post_total_vol, 2),
        'Volume removed': round(removed_total_vol, 2),
        'Pre-thinning median DBH': round(pre_median, 2) if not np.isnan(pre_median) else np.nan,
        'Pre-thinning mean DBH': round(pre_mean, 2)     if not np.isnan(pre_mean)   else np.nan,
        'Post-thinning median DBH': round(post_median, 2) if not np.isnan(post_median) else np.nan,
        'Post-thinning mean DBH': round(post_mean, 2)     if not np.isnan(post_mean)   else np.nan,
        'Change in median DBH': round(d_median, 2) if not np.isnan(d_median) else np.nan,
        'Change in mean DBH':   round(d_mean, 2)   if not np.isnan(d_mean)   else np.nan,
    }

def build_comp_for_k(df, k:int, metric='pre_DBH', vol_col='pre_stem_vol',
                     row_col='Row', status_col='status'):
    rows = []
    for s in range(1, k+1):
        d = k_row_thinning(df, k, s, row_col=row_col, status_col=status_col)
        a = stand_analysis(d, metric=metric, vol_col=vol_col, status_col=status_col)
        a['Strategy']  = f'{k}-row start={s}'
        a['k']         = k
        a['start_row'] = s
        rows.append(a)
    comp = pd.DataFrame(rows)
    for c in ('Q4 retention ratio (post/pre)', 'Q1 removal ratio (removed/pre)'):
        if c in comp.columns: comp[c] = pd.to_numeric(comp[c], errors='coerce')
    return comp

from functools import lru_cache

def variable_row_thinning(df, cut_rows, row_col='Row', status_col='status'):
    d = df.copy()
    d['thin_decision'] = 'Dead (ignored)'
    alive = d[status_col].eq('Alive')
    in_cut = d[row_col].isin(cut_rows)
    d.loc[alive & in_cut,  'thin_decision'] = 'Thin'
    d.loc[alive & ~in_cut, 'thin_decision'] = 'Keep'
    return d

def _row_q4_volume_by_row(df, metric='pre_DBH', vol_col='pre_stem_vol',
                          row_col='Row', status_col='status'):
    alive = df[df[status_col] == 'Alive'].copy()
    if alive.empty:
        raise ValueError("No Alive trees found; cannot compute Q4 volumes.")
    q3 = float(alive[metric].quantile(0.75))
    rows = np.sort(pd.unique(df[row_col]))
    q4_rows = alive.loc[alive[metric] >= q3, [row_col, vol_col]]
    q4_vol_by_row = q4_rows.groupby(row_col)[vol_col].sum()
    q4_vol_by_row = q4_vol_by_row.reindex(rows, fill_value=0.0)
    return rows, q3, q4_vol_by_row.values

def _best_sequence_from_start_q4vol(rows, q4_vols, start_idx, target_cuts, steps=(3,4,5)):
    """
    DP objective: MINIMIZE total Q4 VOLUME removed.
    Tie-breaks: smaller step, then earlier row index.
    Returns (sum_q4_vol, path_indices) or None if infeasible.
    """
    N = len(rows)
    if target_cuts <= 0 or start_idx < 0 or start_idx >= N:
        return None
    if start_idx + 3*(target_cuts-1) > N-1:
        return None

    @lru_cache(maxsize=None)
    def dp(last_idx, selected):
        if selected == target_cuts:
            return (0.0, ())
        remaining = target_cuts - selected
        if last_idx + 3*remaining > N-1:
            return None
        best = None
        cand = []
        for st in steps:
            nxt = last_idx + st
            if nxt <= N-1:
                cand.append((q4_vols[nxt], st, nxt))
        cand.sort(key=lambda x: (x[0], x[1], x[2]))
        for q4v, st, nxt in cand:
            rem_after = target_cuts - (selected + 1)
            if nxt + 3*rem_after > N-1:
                continue
            sub = dp(nxt, selected + 1)
            if sub is None:
                continue
            sub_q4, sub_path = sub
            cand_val = (q4v + sub_q4, (nxt,) + sub_path)
            if best is None or (cand_val[0] < best[0]) or (cand_val[0] == best[0] and cand_val[1] < best[1]):
                best = cand_val
        return best

    start_cost = float(q4_vols[start_idx])
    sub = dp(start_idx, 1)
    if sub is None:
        return None
    sub_q4, sub_path = sub
    total_q4 = start_cost + sub_q4
    path = (start_idx,) + sub_path
    return (total_q4, path)

def choose_variable_cut_rows_q4volume(df, target_cuts:int, metric='pre_DBH', vol_col='pre_stem_vol',
                                      row_col='Row', status_col='status',
                                      first_start_rows:int=5,
                                      min_in_between:int=2, max_in_between:int=4):
    assert min_in_between == 2 and max_in_between == 4, "This version fixes steps to {3,4,5}."
    rows, q3, q4_vols = _row_q4_volume_by_row(df, metric=metric, vol_col=vol_col,
                                              row_col=row_col, status_col=status_col)
    N = len(rows)
    if N == 0:
        return []

    max_start_idx = min(first_start_rows, N) - 1
    feasible_starts = [s for s in range(0, max_start_idx + 1) if s + 3*(target_cuts-1) <= N-1]
    if not feasible_starts:
        raise ValueError(f"Infeasible: cannot place {target_cuts} cuts starting within first {first_start_rows} rows.")

    best_total = None
    best_path  = None
    best_start = None
    for s in feasible_starts:
        res = _best_sequence_from_start_q4vol(rows, q4_vols, s, target_cuts, steps=(3,4,5))
        if res is None:
            continue
        tot_q4, path = res
        if (best_total is None) or (tot_q4 < best_total) or (tot_q4 == best_total and s < best_start):
            best_total = tot_q4
            best_path  = path
            best_start = s

    if best_path is None:
        raise ValueError("No feasible sequence found.")
    return rows[list(best_path)].tolist()

def variable_thinning_variants_volume_pure(df, metric='pre_DBH', vol_col='pre_stem_vol',
                                           row_col='Row', status_col='status'):

    rows_sorted = np.sort(pd.unique(df[row_col]))
    R = len(rows_sorted)
    targets = {
        '3_row_eqv': R // 3,
        '4_row_eqv': R // 4,
        '5_row_eqv': R // 5,
    }
    out = {}
    for label, m in targets.items():
        cuts = choose_variable_cut_rows_q4volume(
            df, m, metric=metric, vol_col=vol_col, row_col=row_col, status_col=status_col,
            first_start_rows=5, min_in_between=2, max_in_between=4
        )
        d_thin = variable_row_thinning(df, cuts, row_col=row_col, status_col=status_col)
        out[label] = (cuts, d_thin)
    return out

def build_comp_for_variable_variants(variants_dict, metric='pre_DBH', vol_col='pre_stem_vol', status_col='status'):

    rows = []
    for label, (cut_rows, d_thin) in variants_dict.items():
        a = stand_analysis(d_thin, metric=metric, vol_col=vol_col, status_col=status_col)
        a['Strategy']  = label
        a['k']         = 'variable'
        a['start_row'] = cut_rows[0] if len(cut_rows) else np.nan
        rows.append(a)
    comp = pd.DataFrame(rows)
    for c in ('Q4 retention ratio (post/pre)', 'Q1 removal ratio (removed/pre)'):
        if c in comp.columns: comp[c] = pd.to_numeric(comp[c], errors='coerce')
    return comp

def adaptive_practical_band(q4_series: pd.Series, lam: float = 0.5,
                            delta_min: float = 0.002, delta_max: float = 0.012) -> float:
    q4 = pd.to_numeric(q4_series, errors='coerce')
    rng = float(q4.max() - q4.min()) if q4.notna().any() else 0.0
    return float(np.clip(lam * rng, delta_min, delta_max))

def rank_by_lexi_apb_strict(comp: pd.DataFrame,
                            p_col: str = 'Q4 retention ratio (post/pre)',
                            s_col: str = 'Q1 removal ratio (removed/pre)',
                            lam: float = 0.5, delta_min: float = 0.002, delta_max: float = 0.012,
                            delta_override: float | None = None) -> pd.DataFrame:

    df = comp.copy()
    p = pd.to_numeric(df[p_col], errors='coerce')
    s = pd.to_numeric(df[s_col], errors='coerce')

    delta  = float(delta_override) if delta_override is not None else adaptive_practical_band(p, lam, delta_min, delta_max)
    p_best = float(p.max())
    close  = (p_best - p) <= delta
    df['_apb_close'] = close

    def _norm(x):
        xmin, xmax = float(x.min()), float(x.max())
        return (x - xmin) / (xmax - xmin) if xmax > xmin else pd.Series(0.5, index=x.index)

    df['Lexi-APB index'] = close.astype(float) + np.where(close, _norm(s), _norm(p)) * 1e-3

    df = (df.sort_values(by=['_apb_close', s_col, p_col],
                         ascending=[False,      False,  False])
            .drop(columns=['_apb_close'])
            .reset_index(drop=True))
    df.attrs['apb_delta'] = delta
    return df

def style_comp_table(comp_ranked: pd.DataFrame, title=None):
    show = [
        'Strategy',
        'Lexi-APB index',
        '% trees removed','% volume removed',
        'Trees kept','Trees removed',
        'Q1 cut (count)','Q1 removal ratio (removed/pre)',
        'Q4 remaining (count)','Q4 cut (count)',
        'Q4 retention ratio (post/pre)',
        '% of removed volume from Q4','Removed volume from Q4',
        'Volume removed','Post-thinning total volume',
        'start_row'
    ]
    cols = [c for c in show if c in comp_ranked.columns]
    tbl  = comp_ranked[cols].copy()

    count_cols = ['Trees kept','Trees removed','Q1 cut (count)','Q4 cut (count)','Q4 remaining (count)']
    pct_cols   = ['% trees removed','% volume removed','% of removed volume from Q4']
    ratio_cols = ['Q4 retention ratio (post/pre)','Q1 removal ratio (removed/pre)']
    money_cols = ['Removed volume from Q4','Volume removed','Post-thinning total volume']

    sty = (tbl.reset_index(drop=True)
           .style
           .format({c:'{:,.0f}' for c in count_cols if c in tbl})
           .format({c:'{:.2f}%' for c in pct_cols   if c in tbl})
           .format({c:'{:.3f}'  for c in ratio_cols if c in tbl})
           .format({c:'{:,.6f}' for c in ['Lexi-APB index (strict)'] if c in tbl})
           .format({c:'{:,.2f}' for c in money_cols if c in tbl})
           .set_caption(title or 'Strategies ranked — Lexi-APB (strict)')
           .set_properties(**{'font-variant-numeric':'tabular-nums'})
           .hide(axis='index'))
    return sty

def score_krow_lexi_apb(df, k:int, metric='pre_DBH', vol_col='pre_stem_vol',
                        row_col='Row', status_col='status',
                        lam:float=0.5, delta_min:float=0.002, delta_max:float=0.012,
                        delta_override:float=None, title=None, return_comp=False):
    comp = build_comp_for_k(df, k, metric=metric, vol_col=vol_col, row_col=row_col, status_col=status_col)
    ranked = rank_by_lexi_apb_strict(comp,
                                     p_col='Q4 retention ratio (post/pre)',
                                     s_col='Q1 removal ratio (removed/pre)',
                                     lam=lam, delta_min=delta_min, delta_max=delta_max,
                                     delta_override=delta_override)
    apb = ranked.attrs.get('apb_delta', None)
    title = title or f'{k}-row — Lexi-APB (δ={apb:.4f})'
    sty = style_comp_table(ranked, title)
    if return_comp:
        return sty, ranked
    return sty

def score_variable_variants_lexi_apb(df, metric='pre_DBH', vol_col='pre_stem_vol',
                                     row_col='Row', status_col='status',
                                     lam:float=0.5, delta_min:float=0.002, delta_max:float=0.012,
                                     delta_override:float=None, title=None, return_comp=False):
    variants = variable_thinning_variants_volume_pure(df, metric=metric, vol_col=vol_col,
                                                      row_col=row_col, status_col=status_col)
    comp = build_comp_for_variable_variants(variants, metric=metric, vol_col=vol_col, status_col=status_col)
    ranked = rank_by_lexi_apb_strict(comp,
                                     p_col='Q4 retention ratio (post/pre)',
                                     s_col='Q1 removal ratio (removed/pre)',
                                     lam=lam, delta_min=delta_min, delta_max=delta_max,
                                     delta_override=delta_override)
    apb = ranked.attrs.get('apb_delta', None)
    title = title or f'Variable thinning — Lexi-APB (δ={apb:.4f})'
    sty = style_comp_table(ranked, title)
    if return_comp:
        return sty, ranked, variants
    return sty

sty5, ranked5 = score_krow_lexi_apb(df, k=5, return_comp=True)
display(sty5); print('Winner (5-row): start_row =', int(ranked5.iloc[0]['start_row']))

sty4, ranked4 = score_krow_lexi_apb(df, k=4, return_comp=True)
display(sty4); print('Winner (4-row): start_row =', int(ranked4.iloc[0]['start_row']))

sty3, ranked3 = score_krow_lexi_apb(df, k=3, return_comp=True)
display(sty3); print('Winner (3-row): start_row =', int(ranked3.iloc[0]['start_row']))

sty_var, ranked_var, variants = score_variable_variants_lexi_apb(df, return_comp=True)
display(sty_var); print('Winner (variable):', ranked_var.iloc[0]['Strategy'])


Strategy,Lexi-APB index,% trees removed,% volume removed,Trees kept,Trees removed,Q1 cut (count),Q1 removal ratio (removed/pre),Q4 remaining (count),Q4 cut (count),Q4 retention ratio (post/pre),% of removed volume from Q4,Removed volume from Q4,Volume removed,Post-thinning total volume,start_row
5-row start=2,1.000693,20.680000,20.580000,2393,624,166,0.211000,654,160,0.803000,38.940000,"3,296.46","8,466.46","32,679.84",2
5-row start=1,1.000520,19.690000,19.810000,2423,594,156,0.198000,651,163,0.800000,40.730000,"3,319.44","8,149.61","32,996.69",1
5-row start=5,1.000000,18.500000,18.780000,2459,558,125,0.159000,655,159,0.805000,41.540000,"3,209.87","7,726.53","33,419.78",5
5-row start=4,0.000455,20.780000,20.450000,2390,627,184,0.234000,650,164,0.799000,39.680000,"3,339.39","8,415.52","32,730.79",4
5-row start=3,0.000000,20.350000,20.390000,2403,614,157,0.199000,646,168,0.794000,40.510000,"3,398.22","8,388.19","32,758.12",3


Winner (5-row): start_row = 2


Strategy,Lexi-APB index,% trees removed,% volume removed,Trees kept,Trees removed,Q1 cut (count),Q1 removal ratio (removed/pre),Q4 remaining (count),Q4 cut (count),Q4 retention ratio (post/pre),% of removed volume from Q4,Removed volume from Q4,Volume removed,Post-thinning total volume,start_row
4-row start=2,1.001000,25.390000,24.950000,2251,766,234,0.297000,613,201,0.753000,40.340000,"4,141.84","10,266.18","30,880.13",2
4-row start=3,1.000225,25.320000,25.140000,2253,764,191,0.242000,621,193,0.763000,37.970000,"3,927.18","10,342.17","30,804.13",3
4-row start=1,0.000174,24.560000,24.640000,2276,741,185,0.235000,606,208,0.744000,40.740000,"4,130.99","10,139.76","31,006.54",1
4-row start=4,0.000000,24.730000,25.270000,2271,746,178,0.226000,602,212,0.740000,41.960000,"4,363.37","10,398.19","30,748.11",4


Winner (4-row): start_row = 2


Strategy,Lexi-APB index,% trees removed,% volume removed,Trees kept,Trees removed,Q1 cut (count),Q1 removal ratio (removed/pre),Q4 remaining (count),Q4 cut (count),Q4 retention ratio (post/pre),% of removed volume from Q4,Removed volume from Q4,Volume removed,Post-thinning total volume,start_row
3-row start=3,1.000000,32.520000,32.540000,2036,981,254,0.322000,562,252,0.690000,38.560000,"5,162.02","13,388.63","27,757.68",3
3-row start=1,0.000000,34.410000,34.400000,1979,1038,269,0.341000,527,287,0.647000,41.110000,"5,819.40","14,155.92","26,990.38",1
3-row start=2,0.000349,33.080000,33.060000,2019,998,265,0.336000,539,275,0.662000,41.040000,"5,581.95","13,601.76","27,544.55",2


Winner (3-row): start_row = 3


Strategy,Lexi-APB index,% trees removed,% volume removed,Trees kept,Trees removed,Q1 cut (count),Q1 removal ratio (removed/pre),Q4 remaining (count),Q4 cut (count),Q4 retention ratio (post/pre),% of removed volume from Q4,Removed volume from Q4,Volume removed,Post-thinning total volume,start_row
5_row_eqv,1.000000,19.260000,18.170000,2436,581,173,0.220000,699,115,0.859000,30.550000,"2,283.88","7,476.66","33,669.65",3
3_row_eqv,0.000000,32.810000,31.940000,2027,990,277,0.352000,573,241,0.704000,36.830000,"4,839.36","13,140.93","28,005.37",1
4_row_eqv,0.000684,25.490000,24.240000,2248,769,220,0.279000,659,155,0.810000,31.130000,"3,105.11","9,975.05","31,171.26",3


Winner (variable): 5_row_eqv
